In [0]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

from pyspark.sql import functions as F
from nsw_property import parser

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "nsw_property")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")


LANDING = f"/Volumes/{CATALOG}/{SCHEMA}/landing/sales/"
BRONZE = f"{CATALOG}.{SCHEMA}.bronze_sales_lines"

print("Landing: ", LANDING)
print("Bronze: ", BRONZE)

In [0]:
new_lines = (
    spark.read.text(LANDING + "*.DAT")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

if spark.catalog.tableExists(BRONZE):
    already_loaded = spark.table(BRONZE).select("_source_file").distinct()
    new_lines = new_lines.join(already_loaded, "_source_file", "left_anti")

new_count = new_lines.count()
new_lines.write.mode("append").saveAsTable(BRONZE)

print(f"Shineer nemsen mur: {new_count}")

In [0]:
display(spark.sql(f"""
                  SELECT 
                  _source_file,
                  count(*) as line_count,
                  MIN(_ingested_at) as ingested_at
                  FROM {BRONZE}
                  GROUP BY _source_file"""))